In [ ]:
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd

In [ ]:
### cell 0 ###

course = pd.read_csv(
    "./notebooks/aieducation/what-course-are-you-going-to-take/input/course-reviews-university-of-waterloo/course_data_clean.csv"
)
factor = 300
course = pd.concat([course] * factor, ignore_index=True)
course.info()

In [ ]:
### cell 1 ###

course = course.dropna()

In [ ]:
### cell 2 ###


def extract_filter_features(df, col_name, threshold):
    """Extracts features for predicting execution time of filtering operation."""

    num_rows = len(df)
    dtype_str = str(df[col_name].dtype)  # Get dtype as string
    nan_ratio = df[col_name].isna().mean()  # Compute NaN ratio

    # Compute selectivity: fraction of rows passing the filter
    if df[col_name].dropna().empty:
        target_selectivity = 0.0
    else:
        target_selectivity = (df[col_name] < threshold).mean()

    return {
        "num_rows": num_rows,
        "type_str": dtype_str,
        "nan_ratio": nan_ratio,
        "target_selectivity": target_selectivity,
    }


extract_filter_features(course, "num_reviews", 10)

In [ ]:
### cell 3 ###

for i_1 in ["useful", "easy", "liked"]:
    course[i_1] = course[i_1].str.replace("%", "")
    course[i_1] = course[i_1].astype("int")